# Mobility of University Students in Europe
This notebook visualizes the mobility of university students between European countries using a chord diagram.

imports of libraries

In [ ]:
import pandas as pd
import holoviews as hv
from holoviews import opts
import math

hv.extension('bokeh')

Mapping country names to ISO-2 codes

In [ ]:
mapping = {
    'Belgium': 'BE','Bulgaria': 'BG','Czechia': 'CZ','Denmark': 'DK',
    'Germany': 'DE','Estonia': 'EE','Ireland': 'IE','Greece': 'GR',
    'Spain': 'ES','France': 'FR','Croatia': 'HR','Italy': 'IT',
    'Cyprus': 'CY','Latvia': 'LV','Lithuania': 'LT','Luxembourg': 'LU',
    'Hungary': 'HU','Malta': 'MT','Netherlands': 'NL','Austria': 'AT',
    'Poland': 'PL','Portugal': 'PT','Slovakia': 'SK','Finland': 'FI',
    'Romania': 'RO','Sweden': 'SE','Iceland': 'IS','Norway': 'NO',
    'Switzerland': 'CH','United Kingdom': 'GB','North Macedonia': 'MK',
    'Albania': 'AL','Serbia': 'RS','Türkiye': 'TR'
}

load data from Excel and rename first column

In [ ]:
df = pd.read_excel('../../data/europe/data_europe.xlsx') \
       .rename(columns={'University ↓ / Student →': 'Origin'})

get country list and corresponding ISO-2 codes

In [ ]:
countries_full = df.columns[1:].tolist()
countries_iso2 = [mapping[c] for c in countries_full]

create OD matrix and fill missing cells with 0

In [ ]:
od = df.set_index('Origin').reindex(index=countries_full, columns=countries_full, fill_value=0)

create edge list for flows > 50, excluding self-loops

In [ ]:
edges_raw = []
for i, src in enumerate(countries_full):
    for j, dst in enumerate(countries_full):
        w = od.loc[dst, src]
        if w > 50 and i != j:
            edges_raw.append((i, j, w))

compute scaling factor for edge thickness

In [ ]:
max_w = max(w for *_, w in edges_raw)
scale = 8 / max_w

create DataFrame with precomputed line widths

In [ ]:
edges_df = pd.DataFrame([
    {'source': i, 'target': j, 'value': w, 'linewidth': w * scale}
    for i, j, w in edges_raw
])

prepare nodes dataset with ISO-2 labels

In [ ]:
nodes_df = pd.DataFrame({'index': range(len(countries_iso2)), 'label': countries_iso2})
nodes_ds = hv.Dataset(nodes_df, 'index', 'label')

create base chord diagram without internal labels

In [ ]:
base = hv.Chord((edges_df, nodes_ds),
                kdims=['source','target'], vdims=['value','linewidth']).opts(
    opts.Chord(
        cmap='Category20',
        node_color='index',
        edge_color='source',
        edge_line_width='linewidth',
        edge_alpha=0.7,
        node_size=15,
        label_text_font_size='0pt',
        width=800, height=800,
        title=' '
    )
)

extract node positions from bokeh layout

In [ ]:
plot = hv.render(base)
graph_renderer = [r for r in plot.renderers if hasattr(r, 'layout_provider')][0]
node_pos = graph_renderer.layout_provider.graph_layout

calculate radial offset and prepare label positions

In [ ]:
radii = {i: math.hypot(x, y) for i, (x, y) in node_pos.items()}
max_r = max(radii.values())
offset_dist = max_r * 0.10
labels_df = pd.DataFrame([
    {
        'x': node_pos[i][0] + (node_pos[i][0]/radii[i]) * offset_dist,
        'y': node_pos[i][1] + (node_pos[i][1]/radii[i]) * offset_dist,
        'text': countries_iso2[i]
    }
    for i in node_pos
])

create labels and overlay with the chord diagram

In [ ]:
labels = hv.Labels(labels_df, ['x','y'], 'text').opts(
    opts.Labels(
        text_font_size='10pt',
        text_color='black',
        text_align='center',
        text_baseline='middle',
        angle=0,
        background_fill_color='white',
        background_fill_alpha=1.0,
        border_line_color='white',
        border_line_width=1
    )
)

final = base * labels
final